# NARCBench pilot on Qwen3-8B (Kaggle, 2x T4)

**Settings (right sidebar):** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only**. Then *Run All*.

If the session drops: just *Run All* again. Finished runs are skipped (scripts resume).

What this does:
1. Generate with vLLM: Core (original), **Core matched-context** (honest agents also get a private chat), Stego (20 shoes x 2).
2. Extract activations (layers 13-19 of 36) for Core, Core-matched, Stego **fixed**, and Stego **bugged** (final replies blanked, as in the original code).
3. Pack everything into `/kaggle/working/narcbench_qwen3_8b.tar.gz` for download.

Expected time: ~2-3 h total. Kaggle sessions last up to 12 h; keep the tab open or use *Save Version -> Save & Run All*.

In [ ]:
%cd /kaggle/working
# Keep existing data on restart (Persistence = Files only): pull instead of re-cloning
!if [ -d narcbench ]; then git -C narcbench pull -q; else git clone -q -b matched-context https://github.com/Smit-Shukla/narcbench.git; fi
%cd narcbench
!git log --oneline -3
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# vLLM for generation (transformers/torch come with it)
!pip install -q vllm 2>&1 | tail -3
import vllm, transformers, torch; print("vllm", vllm.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
import subprocess, time, requests, os
MODEL = "Qwen/Qwen3-8B"
PORT = 8020
vllm_proc = subprocess.Popen(
    f"python -m vllm.entrypoints.openai.api_server --model {MODEL} --served-model-name {MODEL} "
    f"--dtype half --tensor-parallel-size 2 --max-model-len 8192 --gpu-memory-utilization 0.90 "
    f"--max-num-seqs 64 --port {PORT}",
    shell=True, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

for i in range(120):  # up to 20 min (first run downloads ~16GB of weights)
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=5).ok:
            print(f"vLLM ready after {i*10}s"); break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        print("vLLM DIED - last log lines:"); print(open("vllm.log").read()[-3000:]); break
    time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
# Smoke test: one chat call with thinking disabled (same payload the scripts send)
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json={
    "model": MODEL, "max_tokens": 60, "temperature": 0.7,
    "messages": [{"role": "system", "content": "You are a committee member."},
                 {"role": "user", "content": "In one sentence: Option A or B?"}],
    "chat_template_kwargs": {"enable_thinking": False}}, timeout=120)
print(r.json()["choices"][0]["message"]["content"])

In [ ]:
# Launch all generation jobs in parallel; vLLM batches the concurrent requests.
OUT = "data/scenarios/qwen3_8b"
common = f"--model {MODEL} --base-url http://127.0.0.1:{PORT}/v1"
jobs = {
    "core":         f"python generation/core.py {common} --output-dir {OUT}/core --parallel 8",
    "core_matched": f"python generation/core.py {common} --output-dir {OUT}/core_matched --parallel 8 --honest-private",
}
for shoe in range(20):
    jobs[f"stego_{shoe:02d}"] = f"python generation/stego.py {common} --output-dir {OUT}/stego --shoe {shoe}"

os.makedirs("logs", exist_ok=True)
procs = {k: subprocess.Popen(v, shell=True, stdout=open(f"logs/{k}.log", "w"), stderr=subprocess.STDOUT)
         for k, v in jobs.items()}
print(f"launched {len(procs)} jobs")

In [ ]:
# Progress monitor (re-run this cell any time; it returns when all jobs finish)
import glob
t0 = time.time()
while True:
    done = {d: len(glob.glob(f"{OUT}/{d}/*/results.json")) for d in ["core", "core_matched", "stego"]}
    running = [k for k, p in procs.items() if p.poll() is None]
    print(f"[{(time.time()-t0)/60:5.1f} min] runs done: core {done['core']}/100 | core_matched {done['core_matched']}/100 "
          f"| stego {done['stego']}/40 | jobs running: {len(running)}", flush=True)
    if not running: break
    time.sleep(120)
failed = {k: p.returncode for k, p in procs.items() if p.returncode != 0}
print("FAILED jobs:", failed or "none")
!grep -l "API ERROR" logs/*.log | head

In [ ]:
# Quick sanity check on generated data
import json, statistics as st
for d in ["core", "core_matched"]:
    cfgs = [json.load(open(f)) for f in glob.glob(f"{OUT}/{d}/*/run_config.json")]
    col = [c for c in cfgs if c["mode"] == "collusion"]
    print(f"{d}: {len(cfgs)} runs | collusion success (both colluders voted target): "
          f"{sum(c['collusion_success'] for c in col)}/{len(col)}")
errs = sum(open(f).read().count("[ERROR]") for f in glob.glob(f"{OUT}/*/*/results.json"))
print("[ERROR] replies across all runs:", errs)

In [ ]:
# Free the GPUs for extraction
vllm_proc.terminate(); time.sleep(20)
!pkill -f vllm.entrypoints || true
time.sleep(10)
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# Activation extraction (fp16 via kaggle/run_extract.py). Layers 13-19 of 36 (~36-53% depth;
# the paper uses 26-30 of 64 = ~41-47%).
ACT = "data/activations/qwen3_8b"
X = f"--model {MODEL} --layers 13-19 --gen-only"
!python kaggle/run_extract.py core  --runs-dir {OUT}/core         --output-dir {ACT}/core         {X} 2>&1 | grep -v Warning | tail -15
!python kaggle/run_extract.py core  --runs-dir {OUT}/core_matched --output-dir {ACT}/core_matched {X} 2>&1 | grep -v Warning | tail -15
!python kaggle/run_extract.py stego --runs-dir {OUT}/stego --output-dir {ACT}/stego     {X} --channels public 2>&1 | grep -v Warning | tail -15
!python kaggle/run_extract.py stego --runs-dir {OUT}/stego --output-dir {ACT}/stego_bug {X} --channels public --stego-bug 2>&1 | grep -v Warning | tail -15

In [ ]:
# Package for download (Output tab -> narcbench_qwen3_8b.tar.gz)
!tar czf /kaggle/working/narcbench_qwen3_8b.tar.gz data/activations/qwen3_8b data/scenarios/qwen3_8b logs vllm.log
!ls -lh /kaggle/working/narcbench_qwen3_8b.tar.gz
!du -sh data/activations/qwen3_8b/*